# Figure GS03 (supplementary): candidate burden per genome against genome covariates (GTDB r232)

**Status:** draft, Gadi. The ID is provisional. This replaces `Validation/gtdb/figures/candidate_burden_{bac,arc}.png` (genome_entropy #107).

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figGS03_burden_bins.tsv.gz`, `data/figGS03_burden_rho.tsv.gz` | Gadi `missed_genes/candidate_burden_{bac,arc}.tsv`: per-genome candidate counts joined to the GTDB r232 `bac120`/`ar53` metadata | `16_candidate_burden.py`; summaries as `36_candidate_burden_figures.py` |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only GS03`.
  - Each covariate is cut into up to 20 equal-count bins. Each bin carries the 10/25/50/75/90% quantiles and the mean of the burden.
  - Spearman ρ is computed over all genomes, not over the bins.
- **Genomes:** the annotated representatives, 96,875 bacterial and 5,492 archaeal.
- **Independence from the comparator:** candidates are counted per genome, so this figure does not depend on the comparator.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}


In [ ]:
bb = pd.read_csv(DATA / "figGS03_burden_bins.tsv.gz", sep="\t")
rho = pd.read_csv(DATA / "figGS03_burden_rho.tsv.gz", sep="\t")
COVS = [("n50_contigs", "contig N50 (bp)", True), ("contig_count", "contigs", True),
        ("genome_size_mb", "genome\nsize (Mb)", False), ("coding_density", "coding\ndensity (%)", False),
        ("frac_orfs_in_genbank", "fraction of ORFs\nmatched to a CDS", False),
        ("checkm2_completeness", "CheckM2\ncompleteness (%)", False),
        ("checkm2_contamination", "CheckM2\ncontamination (%)", False)]
fig, axes = plt.subplots(2, len(COVS), figsize=(7.2, 3.4), sharey="row")
for i, dom in enumerate(("bacteria", "archaea")):
    for j, (cov, lab, logx) in enumerate(COVS):
        ax = axes[i, j]
        t = bb[(bb.domain == dom) & (bb.covariate == cov) & (bb.burden == "n_candidates")]
        ax.fill_between(t.x_median, t.q10, t.q90, color="#CCE3F2", lw=0)
        ax.fill_between(t.x_median, t.q25, t.q75, color="#7FB8DE", lw=0)
        ax.plot(t.x_median, t.q50, color="#0072B2", lw=1.0, marker="o", ms=1.8)
        ax.plot(t.x_median, t["mean"], color=INK, lw=0.6, ls="--")
        if logx:
            ax.set_xscale("log")
        ax.set_yscale("symlog", linthresh=1)
        ax.set_ylim(0, None)
        r = rho[(rho.domain == dom) & (rho.covariate == cov)].iloc[0]
        ax.set_title(f"ρ = {r.rho_per_genome:+.2f}", fontsize=6.4, loc="left", pad=2)
        ax.tick_params(labelsize=5.6)
        if i == 1:
            ax.set_xlabel(lab, fontsize=6)
        if j == 0:
            ax.set_ylabel(f"{dom}\ncandidates per genome", fontsize=6.4)
panel(axes[0, 0], "a")
panel(axes[1, 0], "b")
fig.tight_layout(w_pad=0.4, h_pad=0.8)
save(fig, "figGS03_candidate_burden")


In [ ]:
print(rho.round(3).to_string(index=False))


## Caption (draft)

**Figure GS03. Candidate burden per genome scales with genome size, not with assembly quality.**

Each annotated GTDB r232 representative contributes its number of candidates. **(a)** Bacteria, 96,875 genomes and 523,346 candidates, median 3 per genome. **(b)** Archaea, 5,492 genomes and 22,447 candidates, median 2 per genome.

Each covariate is cut into ≤20 equal-count bins. The line shows the median burden per bin, the bands the 25–75% and 10–90% ranges, and the dashed line the mean. Spearman ρ, computed over all genomes, is printed above each panel.

- **Genome size:** burden rises with genome size (ρ = 0.55 and 0.51).
- **Annotation completeness:** burden falls with the fraction of ORFs that match a deposited CDS (ρ = −0.43 and −0.38).
- **Assembly quality:** burden is nearly flat against contig N50 (ρ = 0.10 and 0.14).

Per-Mb ρ values are in `data/figGS03_burden_rho.tsv.gz`.
